# Population-weighted distance-decay distribution
First run [3_population_distribution.ipynb](3_population_distribution.ipynb) through its aggregate export section. Run this notebook with the repository root as the working directory.

Workplace-area/residence-region OD targets constrain allocation. Within each pairing, workplace probabilities are proportional to remaining job population times `exp(-distance / DISTANCE_SCALE)`, with origins sampled by remaining residents. Capacities and OD quotas are consumed exactly after rounding/balancing.

In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import geopandas as gpd
import pandas as pd

from src import data_filepaths
from src.utils.loaders import load_boundary_pln_area, load_boundary_pln_region
from src.utils.population_distribution import (
    locate_demand_points, distribute_population, build_demand_data,
)

In [2]:
ALLOW_EXPORT = True

## Configuration
Dense pairs use the large size only if **both** endpoint populations exceed it and their remaining capacities and OD quota can accommodate it. All other groups use the small size. Large must be an integer multiple of small.

Counts are rounded to small-size multiples; the larger resident/job total is reduced proportionally to the smaller using largest remainders. Zero OD weights are soft targets, matching the legacy quota allocator. Same-point commutes are allowed. Driving metrics are straight-line estimates, not road routes.

In [8]:
SMALL_SIZE = 50
LARGE_SIZE = 200
DISTANCE_SCALE = 10_000  # metres, EPSG:3414
RANDOM_SEED = 0
DRIVING_SPEED = 40 / 3.6  # metres per second

OUTPUT_PATH = Path(data_filepaths.OUT_DEMAND_DATA) / 'demand_data_v2.json'

In [4]:
demand_gdf = gpd.read_parquet(data_filepaths.AGGREGATED_DEMAND_POINTS)
df_ewarr = pd.read_parquet(data_filepaths.WORKPLACE_RESIDENCE_OD)
pln_area_gdf = load_boundary_pln_area()
pln_region_gdf = load_boundary_pln_region()
points_gdf = locate_demand_points(demand_gdf, pln_area_gdf, pln_region_gdf)

outside = points_gdf[['pln_area_id', 'pln_region_id']].isna().any(axis=1)
print(f'Excluded {outside.sum()} points outside planning boundaries:')
print(points_gdf.loc[outside, ['residents', 'workers']].sum())
points_gdf = points_gdf.loc[~outside].copy()
points_gdf[['residents', 'workers']].sum()

Excluded 1 points outside planning boundaries:
residents    199.929035
workers        0.000000
dtype: float64


residents    3.018854e+06
workers      3.013468e+06
dtype: float64

In [9]:
allocated_points, pops_df, od_people = distribute_population(
    points_gdf, df_ewarr.loc[:, pln_region_gdf.index],
    small_size=SMALL_SIZE, large_size=LARGE_SIZE,
    distance_scale=DISTANCE_SCALE, seed=RANDOM_SEED,
)
print(pd.DataFrame({
    'input': points_gdf[['residents', 'workers']].sum(),
    'allocated': allocated_points[['residents', 'workers']].sum(),
}))
print('Group sizes:', pops_df['size'].value_counts().to_dict())
od_people.head()

                  input  allocated
residents  3.018854e+06    3013150
workers    3.013468e+06    3013150
Group sizes: {200: 13162, 50: 7615}


,CR,ER,NER,NR,WR
AM,11300,7600,27850,16700,9900
BD,15300,40950,19300,8500,8400
BK,8400,3450,7950,7550,28500
BL,2950,1750,2100,2750,12450
BM,43450,20950,31400,14800,35350


In [ ]:
demand_data = build_demand_data(allocated_points, pops_df, driving_speed=DRIVING_SPEED)
assert sum(p['residents'] for p in demand_data['points']) == sum(p['size'] for p in demand_data['pops'])
assert sum(p['jobs'] for p in demand_data['points']) == sum(p['size'] for p in demand_data['pops'])

In [10]:
if ALLOW_EXPORT:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("w", encoding="utf-8") as output:
        json.dump(demand_data, output, allow_nan=False)
    print(
        f"Exported {len(demand_data['points'])} points and {len(demand_data['pops'])} groups to {OUTPUT_PATH}"
    )

Exported 4230 points and 20777 groups to out\demand_data\demand_data_v2.json
